## BÁO CÁO KỸ THUẬT: PHƯƠNG PHÁP HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH FINE-TUNE (FINE-TUNE ONLY)

## 1. THƯ VIỆN SỬ DỤNG VÀ THÔNG SỐ CẤU HÌNH

### Các thư viện chính
- **transformers & bitsandbytes**: Tải mô hình nền, tokenizer và cấu hình QLoRA 4-bit (NF4, Float16 compute).
- **peft**: Cấu hình và quản lý adapter LoRA.
- **trl (SFTTrainer)**: Quản lý và thực thi quá trình Supervised Fine-Tuning.
- **datasets**: Xử lý, tải trực tiếp dữ liệu từ Hugging Face Hub (`TinPhan2007/vietnam-legal-qa-processed`).
- **rouge_score & tqdm**: Đánh giá chỉ số ROUGE-L và trực quan hóa tiến trình.

### Bảng thông số cấu hình chi tiết

| Giai đoạn | Tham số | Giá trị | Giải thích |
| :--- | :--- | :--- | :--- |
| **Model Base** | `model_name` | `Qwen/Qwen2.5-3B-Instruct` | Mô hình nền 3B tham số, lượng tử hóa 4-bit NF4 (`load_in_4bit = True`, compute dtype `torch.float16`). |
| **LoRA Config** | `r / lora_alpha` | `64 / 128` | LoRA rank và hệ số scaling factor. |
| | `target_modules` | `q, k, v, o, gate, up, down_proj` | Áp dụng LoRA vào toàn bộ các lớp Attention và MLP. |
| **Training** | `max_seq_length` | `1024` | Chiều dài chuỗi đầu vào tối đa khi huấn luyện. |
| | `per_device_train_batch_size` | `2` | Batch size trên mỗi step của GPU. |
| | `gradient_accumulation_steps` | `2` | Tích lũy gradient (Batch size hiệu dụng = 2 × 4 = 8). |
| | `learning_rate / epochs` | `1.5e-4 / 1` | Tốc độ học $2 \times 10^{-4}$ và huấn luyện trong 1 epoch. |
| **Inference** | `BATCH_SIZE` | `16` | Xử lý song song 8 mẫu cùng lúc trên GPU. |
| | `padding_side` | `"left"` | Đặt padding bên trái phục vụ batch inference. |
| | `max_new_tokens` | `512` | Giới hạn độ dài sinh token của câu trả lời. |
---

In [ ]:
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 39.5 MB/s eta 0:00:00


In [ ]:
!pip install -q rouge-score trl

In [ ]:
pip install rouge-score tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os

# 1. TẮT HOÀN TOÀN GRADSCALER CỦA ACCELERATE ĐỂ TRÁNH LỖI BFLOAT16
os.environ.pop("ACCELERATE_MIXED_PRECISION", None)
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

accelerate_config_path = os.path.expanduser("~/.cache/huggingface/accelerate/default_config.yaml")
if os.path.exists(accelerate_config_path):
    os.remove(accelerate_config_path)

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

assert torch.cuda.device_count() == 1, f"Vẫn thấy {torch.cuda.device_count()} GPU, cần restart kernel trước khi chạy cell này"

HF_REPO = "l3mon3/Vietnamese_legal_dataset"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_ADAPTER = "/kaggle/working/qwen_legal_lora_rag_ft"
max_seq_length = 1536

# T4 sử dụng fp16 cho compute dtype để kích hoạt Tensor Cores
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
    trust_remote_code=True,
)
model.config.torch_dtype = torch.float16

model = prepare_model_for_kbit_training(
    model,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)
model = get_peft_model(model, peft_config)

# ==========================================
# FORMAT DỮ LIỆU TRAIN & VAL
# ==========================================
def format_rag_sample(sample):
    docs = sample.get("retrieved_docs") or []
    context_blocks = []
    for i, doc in enumerate(docs[:2], start=1):
        context_blocks.append(f"--- [Tài liệu {i}] ---\n{doc.get('document', '')}")
    context_str = "\n\n".join(context_blocks)
    user_prompt = f"""Bạn là một chuyên gia tư vấn pháp luật. Dựa trên các quy định của pháp luật Việt Nam được cung cấp dưới đây, hãy giải đáp câu hỏi của người dùng và BẮT BUỘC trình bày theo đúng định dạng chuẩn 3 phần sau:

**Căn cứ pháp lý:** [Tên điều luật] - [Tên văn bản luật]

**Nội dung quy định:**
[Trích dẫn chính xác nội dung điều luật áp dụng]

**Phân tích & Hướng dẫn:**
[Phân tích, áp dụng quy định trên vào tình huống của người dùng để trả lời câu hỏi]

---
CĂN CỨ PHÁP LUẬT THAM KHẢO:
{context_str}

CÂU HỎI:
{sample.get('query', '')}

TRẢ LỜI:"""
    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": sample.get("target_response", "")}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

def format_val_with_synthetic_target(sample):
    law_id = sample.get("ground_truth_law_id", "")
    law_content = sample.get("ground_truth_law_content", "")
    query = sample.get("query", "")

    synthetic_target = f"""**Căn cứ pháp lý:** {law_id}

**Nội dung quy định:**
{law_content}

**Phân tích & Hướng dẫn:**
Căn cứ vào quy định nêu trên, đối với thắc mắc "{query}", vấn đề này được điều chỉnh và áp dụng trực tiếp theo các nguyên tắc, phạm vi của {law_id}."""

    docs = sample.get("retrieved_docs") or []
    context_blocks = []
    for i, doc in enumerate(docs[:2], start=1):
        context_blocks.append(f"--- [Tài liệu {i}] ---\n{doc.get('document', '')}")
    context_str = "\n\n".join(context_blocks)

    user_prompt = f"""Bạn là một chuyên gia tư vấn pháp luật. Dựa trên các quy định của pháp luật Việt Nam được cung cấp dưới đây, hãy giải đáp câu hỏi của người dùng và BẮT BUỘC trình bày theo đúng định dạng chuẩn 3 phần sau:

**Căn cứ pháp lý:** [Tên điều luật] - [Tên văn bản luật]

**Nội dung quy định:**
[Trích dẫn chính xác nội dung điều luật áp dụng]

**Phân tích & Hướng dẫn:**
[Phân tích, áp dụng quy định trên vào tình huống của người dùng để trả lời câu hỏi]

---
CĂN CỨ PHÁP LUẬT THAM KHẢO:
{context_str}

CÂU HỎI:
{query}

TRẢ LỜI:"""

    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": synthetic_target}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# ==========================================
# NẠP VÀ LỌC DATASET
# ==========================================
train_raw = load_dataset(HF_REPO, data_files="train.parquet", split="train")
eval_raw = load_dataset(HF_REPO, data_files="validation.parquet", split="train")

train_dataset = train_raw.map(format_rag_sample, remove_columns=train_raw.column_names)
train_dataset = train_dataset.filter(
    lambda x: len(tokenizer(x["text"], truncation=False)["input_ids"]) <= max_seq_length,
    num_proc=2
)
print(f">>> Tập Train sạch (<= {max_seq_length}): {len(train_dataset)} mẫu")

eval_dataset_formatted = eval_raw.map(format_val_with_synthetic_target, remove_columns=eval_raw.column_names)
eval_dataset = eval_dataset_formatted.filter(
    lambda x: len(tokenizer(x["text"], truncation=False)["input_ids"]) <= max_seq_length,
    num_proc=2
)
eval_dataset = eval_dataset.select(range(min(100, len(eval_dataset))))
print(f">>> Tập Validation dùng để eval: {len(eval_dataset)} mẫu")

# ==========================================
# THIẾT LẬP HUẤN LUYỆN
# ==========================================
training_args = SFTConfig(
    output_dir="/kaggle/working/qwen_legal_ft_outputs",
    dataset_text_field="text",
    max_length=max_seq_length,
    packing=False,
    group_by_length=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    warmup_steps=20,
    num_train_epochs=2,
    learning_rate=1.5e-4,
    fp16=False,                          # Tắt GradScaler để loại bỏ triệt để lỗi unscale
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

trainer.model.save_pretrained(OUTPUT_ADAPTER)
tokenizer.save_pretrained(OUTPUT_ADAPTER)
print(f">>> Đã huấn luyện xong và lưu adapter tại: {OUTPUT_ADAPTER}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.183187,0.371448,0.318018,2789576.000000,0.910822
2,0.093986,0.316708,0.211070,5579152.000000,0.933477


>>> Đã huấn luyện xong và lưu adapter tại: /kaggle/working/qwen_legal_lora_rag_ft
